In [ ]:
file1 = '../local_snapshot/data/uniprotkb_AND_reviewed_true_2023_07_03.tsv'
file3 = '../local_snapshot/data/all/all_cluster.tsv'
file4 = '../local_snapshot/data/all/afall_ID50_cluster.tsv'
file5 = '../local_snapshot/data/all/select_cluster.tsv'
from CLEAN.utils import *
dirs = "../local_snapshot/tttianhao-CLEAN-0cf2cac/"
id_ec, ec_id =  get_ec_id_dict(dirs+'data/split100' + '.csv')

In [ ]:
import pandas as pd
GO_table = pd.read_csv(file1, sep='\t')
GO_table

In [ ]:
GO_table[GO_table['Entry']=='A0A009IHW8']['Gene Ontology (GO)'].values.tolist()

In [ ]:
rep_seq = {}
rep_fold = {}
seqfile = open(file3,'r')
for line in seqfile:
    key = line.split('\t')[0]
    if key in rep_seq.keys():
        rep_seq[key].append(line.split('\t')[1].split('\n')[0])
    else:
        rep_seq[key] = []
        rep_seq[key].append(line.split('\t')[1].split('\n')[0])
        
foldfile = open(file5,'r')
for line in foldfile:
    key = line.split('\t')[0].split('.')[0]
    if key in rep_fold.keys():
        rep_fold[key].append(line.split('\t')[1].split('.')[0])
    else:
        rep_fold[key] = []
        rep_fold[key].append(line.split('\t')[1].split('.')[0])

In [ ]:
import os
import numpy as np
def get_pLDDT(pdb_p):
    infile = open(pdb_p,'r')
    pLDDT = []
    for line in infile:
          if line[0:6].rstrip() == 'ATOM' and line[13:15].rstrip() == 'CA':
            pLDDT.append(float(line[60:65].rstrip()))
    infile.close()
    return np.mean(pLDDT)
af_path = '../local_snapshot/data/afall/'
af_list = os.listdir(af_path)
rep_structure = []
rep_pLDDT = []
keys = []
for key in rep_seq.keys():
    pLDDTs = []
    items = []
    for item in rep_seq[key]:
        item_name = item + '.pdb'
        if item_name in af_list:
            pdb_p = af_path+ item + '.pdb'
            pLDDTs.append(get_pLDDT(pdb_p))
            items.append(item)
    if len(pLDDTs)>0:
        max_plddt = max(pLDDTs)
        rep_pLDDT.append(max_plddt)
        rep_structure.append(items[pLDDTs.index(max_plddt)])
        keys.append(key)

In [ ]:
threshold_value = 90 
positions = [index for index, value in enumerate(rep_pLDDT) if value > threshold_value]

sel_struc=[rep_structure[index ]for index in positions]
sel_keys = [keys[index ]for index in positions]


In [ ]:
fun_fold = {}
sort_items = []
sel_seq = {}
for key in rep_fold.keys():
    fun_fold[key] = []
    
    for item in rep_fold[key]:
        if item in sel_struc:
            k = sel_keys[sel_struc.index(item)]
            fun_fold[key].extend(rep_seq[k])
            sel_seq[k] = rep_seq[k]
        
            sort_items.extend(rep_seq[k])
        #print(key,item,len(sort_items))
        if item not in sel_struc and len(rep_fold[key])==1:
            fun_fold[key].append(key)
            sort_items.append(key)

In [ ]:
fun_check = {}
fun_key = {}
fold_index = []
fold_items = []
for key in fun_fold.keys():
    fun_key[key] = id_ec[key]
    fun_check[key]= []
    for item in fun_fold[key]:
        fun_check[key].append(id_ec[item])
        fold_index.append(key)
        fold_items.append(item)

fun_clu = {}
for key in ec_id.keys():
    fun_clu[key] = []
    for item in ec_id[key]:
        if item in fold_items:
            fun_clu[key].append(fold_index[fold_items.index(item)])
    if len(fun_clu[key]) == 0:
        fun_clu.pop(key, None)


uni_fold = []
for key in fun_clu.keys():
    list_of_lists=fun_clu[key]
    uni_fold.append(all(sublist == list_of_lists[0] for sublist in list_of_lists))
    
uni_fun = []
for key in fun_check.keys():
    list_of_lists=fun_check[key]
    uni_fun.append(all(sublist == list_of_lists[0] for sublist in list_of_lists))
    


In [ ]:
fun_check_fold = {}
fun_key_fold = {}
fold_index_fold = []
fold_items_fold = []
for key in rep_fold.keys():
    fun_key_fold[key] = id_ec[key]
    fun_check_fold[key]= []
    for item in rep_fold[key]:
        fun_check_fold[key].append(id_ec[item])
        fold_index_fold.append(key)
        fold_items_fold.append(item)

uni_fun_fold = []
for key in fun_check_fold.keys():
    list_of_lists=fun_check_fold[key]
    uni_fun_fold.append(all(sublist == list_of_lists[0] for sublist in list_of_lists))

In [ ]:
sel_seq = {}
for item in sel_struc:
    k = sel_keys[sel_struc.index(item)]
    sel_seq[k] = rep_seq[k]
    
members = []
keys = []
for key in sel_seq.keys():
    keys.append(key)
    members.append(len(sel_seq[key]))
    
sing_seq = [keys[index] for index, value in enumerate(members) if value == 1]

members = []
keys = []
for key in rep_fold.keys():
    keys.append(key)
    members.append(len(rep_fold[key]))
sing_fold = [keys[index] for index, value in enumerate(members) if value == 1]

members = []
keys = []
for key in fun_fold.keys():
    keys.append(key)
    members.append(len(fun_fold[key]))
sing_funfold = [keys[index] for index, value in enumerate(members) if value == 1]

members = []
keys = []
for key in fun_clu.keys():
    keys.append(key)
    members.append(len(fun_clu[key]))
sing_fold_fun = [keys[index] for index, value in enumerate(members) if value == 1]

In [ ]:
keys = []
for key in fun_check_fold.keys():
    keys.append(key)
uni_fun_fold_keys = []
uni_nofun_fold_keys = []
for i in range(len(keys)):
    if uni_fun[i]:
        uni_fun_fold_keys.append(keys[i])
    else:
        uni_nofun_fold_keys.append(keys[i])

In [ ]:
len(uni_fun_fold_keys)

In [ ]:
len(uni_nofun_fold_keys)

In [ ]:
uni_nofun3_fold = []
for key in uni_nofun_fold_keys:
    list_of_lists=fun_check_fold[key]
    uni_nofun3_fold.append(all('.'.join(sublist[0].split('.')[:3]) == '.'.join(list_of_lists[0][0].split('.')[:3]) for sublist in list_of_lists))

In [ ]:
keys = []
for key in uni_nofun_fold_keys:
    keys.append(key)
uni_fun3_fold_keys = []
uni_nofun3_fold_keys = []
for i in range(len(keys)):
    if uni_nofun3_fold[i]:
        uni_fun3_fold_keys.append(keys[i])
    else:
        uni_nofun3_fold_keys.append(keys[i])

In [ ]:
len(uni_fun3_fold_keys)

In [ ]:
len(uni_nofun3_fold_keys)

In [ ]:
(2283+522)/3291

In [ ]:
GO_dic = {}

# Iterate over each row in the DataFrame
for index, row in GO_table.iterrows():
    entry = row['Entry']
    go_terms = row['Gene Ontology (GO)']
    
    # Check if the GO terms are not NaN (missing)
    if not pd.isna(go_terms):
        # Split the GO terms and store them in the dictionary
        GO_dic[entry] = [term.strip() for term in go_terms.split(';')]




In [ ]:
fun_GO = {}
for key in fun_fold.keys():
    if len(fun_fold[key])>10:
        fun_GO[key] = {}
        for item in fun_fold[key]:
            if item in GO_dic.keys():
                fun_GO[key][item]=GO_dic[item]
        if len(fun_GO[key]) == 0:
            fun_GO.pop(key, None)

In [ ]:
GO_mfdic = {}

# Iterate over each row in the DataFrame
for index, row in GO_table.iterrows():
    entry = row['Entry']
    go_terms = row['Gene Ontology (molecular function)']
    
    # Check if the GO terms are not NaN (missing)
    if not pd.isna(go_terms):
        # Split the GO terms and store them in the dictionary
        GO_mfdic[entry] = [term.strip() for term in go_terms.split(';')]


In [ ]:
fun_mfGO = {}
for key in fun_fold.keys():
    if len(fun_fold[key])>10:
        fun_mfGO[key] = {}
        for item in fun_fold[key]:
            if item in GO_mfdic.keys():
                fun_mfGO[key][item]=GO_mfdic[item]
        if len(fun_mfGO[key]) == 0:
            fun_mfGO.pop(key, None)

In [ ]:
funF_mfGO = {}
for key in fun_fold.keys():
    if len(fun_fold[key])>10:
        
        for item in fun_fold[key]:
            if item in GO_mfdic.keys():
                funF_mfGO[item]=GO_mfdic[item]
        if len(funF_mfGO[item]) == 0:
            funF_mfGO.pop(item, None)

In [ ]:
GO_ccdic = {}

# Iterate over each row in the DataFrame
for index, row in GO_table.iterrows():
    entry = row['Entry']
    go_terms = row['Gene Ontology (cellular component)']
    
    # Check if the GO terms are not NaN (missing)
    if not pd.isna(go_terms):
        # Split the GO terms and store them in the dictionary
        GO_ccdic[entry] = [term.strip() for term in go_terms.split(';')]


In [ ]:
fun_ccGO = {}
for key in fun_fold.keys():
    if len(fun_fold[key])>10:
        fun_ccGO[key] = {}
        for item in fun_fold[key]:
            if item in GO_ccdic.keys():
                fun_ccGO[key][item]=GO_ccdic[item]
        if len(fun_ccGO[key]) == 0:
            fun_ccGO.pop(key, None)

In [ ]:
funF_ccGO = {}
for key in fun_fold.keys():
    if len(fun_fold[key])>10:
        
        for item in fun_fold[key]:
            if item in GO_ccdic.keys():
                funF_ccGO[item]=GO_ccdic[item]
        #if len(funF_ccGO[item]) == 0:
            #funF_ccGO.pop(item, None)

In [ ]:
len()

In [ ]:
GO_bpdic = {}

# Iterate over each row in the DataFrame
for index, row in GO_table.iterrows():
    entry = row['Entry']
    go_terms = row['Gene Ontology (biological process)']
    
    # Check if the GO terms are not NaN (missing)
    if not pd.isna(go_terms):
        # Split the GO terms and store them in the dictionary
        GO_bpdic[entry] = [term.strip() for term in go_terms.split(';')]


In [ ]:
fun_bpGO = {}
for key in fun_fold.keys():
    if len(fun_fold[key])>10:
        fun_bpGO[key] = {}
        for item in fun_fold[key]:
            if item in GO_bpdic.keys():
                fun_bpGO[key][item]=GO_bpdic[item]
        if len(fun_bpGO[key]) == 0:
            fun_bpGO.pop(key, None)

In [ ]:
funF_bpGO = {}
for key in fun_fold.keys():
    if len(fun_fold[key])>10:
        
        for item in fun_fold[key]:
            if item in GO_bpdic.keys():
                funF_bpGO[item]=GO_bpdic[item]
        #if len(funF_bpGO[item]) == 0:
            #funF_bpGO.pop(item, None)

In [ ]:
len(funF_bpGO)

In [ ]:
GO_dic['A0A0H3JU78']

In [ ]:
85000/121909

In [ ]:
len(fun_GO)

In [ ]:
import h5py
data = fun_bpGO

with h5py.File('../local_snapshot/data/GO_analysis/fun_bpGO.h5', 'w') as f:
    
    for outer_key, inner_dict in data.items():
        
        group = f.create_group(outer_key)
        
        for inner_key, value in inner_dict.items():
            
            group.create_dataset(inner_key, data=value)


In [ ]:
import h5py
data = fun_mfGO

with h5py.File('../local_snapshot/data/GO_analysis/fun_mfGO.h5', 'w') as f:
    
    for outer_key, inner_dict in data.items():
        
        group = f.create_group(outer_key)
        
        for inner_key, value in inner_dict.items():
            
            group.create_dataset(inner_key, data=value)


In [ ]:
import h5py
data = fun_ccGO

with h5py.File('../local_snapshot/data/GO_analysis/fun_ccGO.h5', 'w') as f:
    
    for outer_key, inner_dict in data.items():
        
        group = f.create_group(outer_key)
        
        for inner_key, value in inner_dict.items():
            
            group.create_dataset(inner_key, data=value)


In [ ]:
import h5py
data = funF_ccGO
with h5py.File('../local_snapshot/data/GO_analysis/funF_ccGO.h5', 'w') as f:
    for key, inner_dict in data.items():    
        f.create_dataset(key, data=value)
        
data = funF_bpGO
with h5py.File('../local_snapshot/data/GO_analysis/funF_bpGO.h5', 'w') as f:
    for key, inner_dict in data.items():    
        f.create_dataset(key, data=value)
        
data = funF_mfGO
with h5py.File('../local_snapshot/data/GO_analysis/funF_mfGO.h5', 'w') as f:
    for key, inner_dict in data.items():    
        f.create_dataset(key, data=value)
        
       

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

#GO_dic = funF_ccGO
GO_dic = fun_bpGO['A8WFT6']
# Count the occurrences of each GO term across all genes
go_term_counts = {}
for gene, go_terms in GO_dic.items():
    for go_term in go_terms:
        if go_term in go_term_counts:
            go_term_counts[go_term] += 1
        else:
            go_term_counts[go_term] = 1

# Total number of genes in the dataset
total_genes = len(GO_dic)

# Compute p-values using hypergeometric test for each GO term
p_values = {}
for go_term, count in go_term_counts.items():
    # Number of genes associated with the GO term
    genes_with_term = count
    # Total number of genes in the population associated with any GO term
    total_genes_with_term = len([gene for gene, terms in GO_dic.items() if go_term in terms])
    # Number of genes not associated with the GO term
    genes_without_term = total_genes - total_genes_with_term
    # Total number of genes not associated with any GO term
    total_genes_without_term = total_genes - genes_with_term
    # Perform hypergeometric test
    p_value = stats.hypergeom.sf(genes_with_term - 1, total_genes, total_genes_with_term, count)
    p_values[go_term] = p_value

# Combine GO term counts and p-values
sorted_go_terms = sorted(go_term_counts.items(), key=lambda x: x[1], reverse=True)

# Print the top N most common GO terms along with their p-values
N = 627  # Number of top GO terms to display
print(f"Top {N} most common GO terms:")
for go_term, count in sorted_go_terms[:N]:
    p_value = p_values.get(go_term, None)
    if p_value is not None:
        print(f"{go_term}: Count={count}, P-value={p_value} (Significant)" if p_value < 0.05 else f"{go_term}: Count={count}, P-value={p_value}")

# Visualize the results (simple bar plot)
top_go_terms = [term[0] for term in sorted_go_terms[:N]]
counts = [term[1] for term in sorted_go_terms[:N]]

plt.figure(figsize=(10, 6))
plt.barh(range(N), counts, color='skyblue')
plt.yticks(range(N), top_go_terms)
plt.xlabel('Count')
plt.ylabel('GO Term')
plt.title('Top Enriched GO Terms')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
85000/121909

In [ ]:
from goatools.go_enrichment import GOEnrichmentStudy
from goatools.obo_parser import GODag

geneid2gos = fun_bpGO['A0A2U7XUH7']
obo_dag = GODag("../local_snapshot/data/go-basic.obo")

# 创建 GO 富集分析器
goeaobj = GOEnrichmentStudy(
    list(geneid2gos.keys()),  # 基因列表
    geneid2gos,  # 基因与 GO 术语的关联数据
    obo_dag,  # GO 数据
    propagate_counts=False,
    alpha=0.05,
    methods=["fdr_bh"]
)

# 执行 GO 富集分析
goea_results = goeaobj.run_study(list(geneid2gos.keys()))

# 打印显著的 GO 术语及其 P 值
for rec in goea_results:
    if rec.p_fdr_bh < 0.05:
        print(rec.GO, rec.name, rec.p_fdr_bh)




In [ ]:
from goatools.gosubdag.plot.gosubdag_plot import plot_gosubdag

# 过滤出显著的GO富集结果
significant_gos = [r.GO for r in goea_results if r.p_fdr_bh < 0.05]

# 创建GO富集结果字典，以便于将显著GO术语的P值添加到GO图中
goea_results_dict = {r.GO: r for r in goea_results}

# 绘制GO图并显示P值
plot_gosubdag(significant_gos, go_dag, goea_results=goea_results_dict, mark_color='pval', dpi=300)


In [ ]:
import matplotlib.pyplot as plt

# 过滤出显著的GO富集结果
significant_gos = [r.GO for r in goea_results if r.p_fdr_bh < 0.05]
p_values = [r.p_fdr_bh for r in goea_results if r.p_fdr_bh < 0.05]

# 绘制柱状图
plt.figure(figsize=(10, 6))
plt.barh(significant_gos, p_values, color='skyblue')
plt.xlabel('P Value')
plt.ylabel('GO Term')
plt.title('Significant GO Enrichment')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
geneid2gos

In [ ]:
goea_results

In [ ]:
from rpy2.robjects.packages import importr



In [ ]:
import pandas as pd
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri

# 将pandas DataFrame转换为R中的数据框
def pandas_to_r_dataframe(df):
    pandas2ri.activate()
    return pandas2ri.py2ri(df)

# 导入R中的clusterProfiler包
clusterProfiler = importr("clusterProfiler")

# 读取基因与GO术语的数据
geneid2gos = {
    'Gene1': ['GO:0001', 'GO:0002', 'GO:0003'],
    'Gene2': ['GO:0002', 'GO:0004'],
    'Gene3': ['GO:0001', 'GO:0003', 'GO:0005'],
    'Gene4': ['GO:0002', 'GO:0005'],
    # 添加更多基因和它们的GO术语
}

# 将基因与GO术语的数据转换为pandas DataFrame
gene_df = pd.DataFrame([(gene, go) for gene, gos in geneid2gos.items() for go in gos], columns=['gene', 'GO'])

# 调用R中的clusterProfiler包进行GO富集分析
enrich_result = clusterProfiler.enrichGO(gene_df, OrgDb="org.Sc.sgd.db", keyType="ENSEMBL")

# 打印GO富集结果
print(enrich_result)


In [ ]:
file_BP = '../local_snapshot/data/GO_analysis/BP_pro.txt'
file_MF = '../local_snapshot/data/GO_analysis/MF_pro.txt'
file_CC = '../local_snapshot/data/GO_analysis/CC_pro.txt'

In [ ]:
data_BP = open(file_BP,'r')
GO_pro_BP = {}
GO_uni_pro_BP = {}
GO_non_pro_BP = {}
for line in data_BP:
    tem = line.split('\t')
    
    GO_pro_BP[tem[0].replace('"', '')] = float(tem[1])
    if tem[0].replace('"', '') in uni_fun_fold_keys:
        GO_uni_pro_BP[tem[0].replace('"', '')] = float(tem[1])
    else:
        GO_non_pro_BP[tem[0].replace('"', '')] = float(tem[1])
    
data_MF = open(file_MF,'r')
GO_pro_MF = {}
GO_uni_pro_MF = {}
GO_non_pro_MF = {}
for line in data_MF:
    tem = line.split('\t')
    
    GO_pro_MF[tem[0].replace('"', '')] = float(tem[1])
    if tem[0].replace('"', '') in uni_fun_fold_keys:
        GO_uni_pro_MF[tem[0].replace('"', '')] = float(tem[1])
    else:
        GO_non_pro_MF[tem[0].replace('"', '')] = float(tem[1])

data_CC = open(file_CC,'r')
GO_pro_CC = {}
GO_uni_pro_CC = {}
GO_non_pro_CC = {}
for line in data_CC:
    tem = line.split('\t')
    
    GO_pro_CC[tem[0].replace('"', '')] = float(tem[1])
    if tem[0].replace('"', '') in uni_fun_fold_keys:
        GO_uni_pro_CC[tem[0].replace('"', '')] = float(tem[1])
    else:
        GO_non_pro_CC[tem[0].replace('"', '')] = float(tem[1])


In [ ]:
max(list(GO_pro_CC.values()))

In [ ]:
# 计算列表中值为 1 的数量
count_ones = sum(1 for value in list(GO_pro_CC.values()) if value == 1)

# 计算列表总长度
total_count = len(list(GO_pro_CC.values()))

# 计算百分比
percentage = (count_ones / total_count) * 100

print(f"The percentage of values equal to 1 in the list is: {percentage:.2f}%")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 两个数据列表
data1 = list(GO_pro_BP.values())
data2 = list(GO_pro_MF.values())
data3 = list(GO_pro_CC.values())
# 使用 Seaborn 绘制两个响铃图在同一个图上
ax = sns.violinplot(data=[data1, data2,data3], palette="muted", orient='v')

#plt.title('Significant GO Enrichment Proportion across Clusters')
#plt.xlabel('Density')
plt.ylabel('Proportion below 0.05')
plt.grid(False)
# 设置 x 轴的刻度和标签
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['BP', 'MF','CC'])
ax.set_ylim(0, 1.02) 
#plt.xticks(rotation=15)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 两个数据列表
data1 = list(GO_uni_pro_BP.values())
data2 = list(GO_non_pro_BP.values())
#data3 = list(GO_pro_CC.values())
# 使用 Seaborn 绘制两个响铃图在同一个图上
ax = sns.violinplot(data=[data1, data2], palette="muted", orient='v')

#plt.title('Significant GO Enrichment Proportion across Clusters')
#plt.xlabel('Density')
plt.ylabel('Proportion below 0.05')
plt.grid(False)
# 设置 x 轴的刻度和标签
ax.set_xticks([0, 1])
ax.set_xticklabels(['EC consistent', 'EC not consistent'])
#plt.xticks(rotation=15)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 分组数据
data_group1 = [list(GO_uni_pro_BP.values()), list(GO_non_pro_BP.values())]
data_group2 = [list(GO_uni_pro_MF.values()), list(GO_non_pro_MF.values())]
data_group3 = [list(GO_uni_pro_CC.values()), list(GO_non_pro_CC.values())]

# 创建一个包含3个子图的图
fig, axes = plt.subplots(1, 3)
titles = ['BP','MF','CC']
# 绘制每个子图的小提琴图
for i, data_group in enumerate([data_group1, data_group2, data_group3]):
    ax = sns.violinplot(data=data_group, palette="muted", orient='v', ax=axes[i])
    #ax.set_xlabel('Proportion below 0.05')
    #ax.set_ylabel('Groups')
    ax.grid(False)
    ax.set_xticklabels(['EC consistent', 'EC not consistent'])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15) 
    #ax.set_yticks([0, 1])
    #ax.set_yticklabels(['Group 1', 'Group 2'])
    ax.set_title(titles[i]) 
    ax.set_ylim(0, 1.02) 
ax=axes[0]
ax.set_ylabel('Proportion below 0.05')
# 调整子图之间的间距
plt.tight_layout()

plt.show()


In [ ]:
file_lenBP = '../local_snapshot/data/GO_analysis/BPlen_pro.txt'
file_lenMF = '../local_snapshot/data/GO_analysis/MFlen_pro.txt'
file_lenCC = '../local_snapshot/data/GO_analysis/CClen_pro.txt'

In [ ]:
data_BP = open(file_lenBP,'r')
GO_pro_BP = {}
GO_uni_pro_BP = {}
GO_non_pro_BP = {}
for line in data_BP:
    tem = line.split('\t')
    
    GO_pro_BP[tem[0].replace('"', '')] = float(tem[1])
    if tem[0].replace('"', '') in uni_fun_fold_keys:
        GO_uni_pro_BP[tem[0].replace('"', '')] = float(tem[1])
    else:
        GO_non_pro_BP[tem[0].replace('"', '')] = float(tem[1])
    
data_MF = open(file_lenMF,'r')
GO_pro_MF = {}
GO_uni_pro_MF = {}
GO_non_pro_MF = {}
for line in data_MF:
    tem = line.split('\t')
    
    GO_pro_MF[tem[0].replace('"', '')] = float(tem[1])
    if tem[0].replace('"', '') in uni_fun_fold_keys:
        GO_uni_pro_MF[tem[0].replace('"', '')] = float(tem[1])
    else:
        GO_non_pro_MF[tem[0].replace('"', '')] = float(tem[1])

data_CC = open(file_lenCC,'r')
GO_pro_CC = {}
GO_uni_pro_CC = {}
GO_non_pro_CC = {}
for line in data_CC:
    tem = line.split('\t')
    
    GO_pro_CC[tem[0].replace('"', '')] = float(tem[1])
    if tem[0].replace('"', '') in uni_fun_fold_keys:
        GO_uni_pro_CC[tem[0].replace('"', '')] = float(tem[1])
    else:
        GO_non_pro_CC[tem[0].replace('"', '')] = float(tem[1])


In [ ]:
max(data3)

In [ ]:
list(GO_pro_BP.values()).index(627)

In [ ]:
list(GO_pro_BP.keys())[132]

In [ ]:
GO_pro_BP['A8WFT6']

In [ ]:
len(fun_fold['A8WFT6'])

In [ ]:
fun_bpGO['A8WFT6']

In [ ]:
GO_dic['A8WFT6']

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 两个数据列表
data1 = list(GO_pro_BP.values())
data2 = list(GO_pro_MF.values())
data3 = list(GO_pro_CC.values())
# 使用 Seaborn 绘制两个响铃图在同一个图上
ax = sns.violinplot(data=[data1, data2,data3], palette="muted", orient='v')

#plt.title('Significant GO Enrichment Proportion across Clusters')
#plt.xlabel('Density')
plt.ylabel('GO term count')
plt.grid(False)
# 设置 x 轴的刻度和标签
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['BP', 'MF','CC'])
#plt.xticks(rotation=15)
#plt.ylim(0, 30) 
plt.show()

In [ ]:
np.mean(list(GO_non_pro_CC.values()))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 分组数据
data_group1 = [list(GO_uni_pro_BP.values()), list(GO_non_pro_BP.values())]
data_group2 = [list(GO_uni_pro_MF.values()), list(GO_non_pro_MF.values())]
data_group3 = [list(GO_uni_pro_CC.values()), list(GO_non_pro_CC.values())]

# 创建一个包含3个子图的图
fig, axes = plt.subplots(1, 3)
titles = ['BP','MF','CC']
# 绘制每个子图的小提琴图
for i, data_group in enumerate([data_group1, data_group2, data_group3]):
    ax = sns.violinplot(data=data_group, palette="muted", orient='v', ax=axes[i])
    #ax.set_xlabel('Proportion below 0.05')
    #ax.set_ylabel('Groups')
    ax.grid(False)
    ax.set_xticklabels(['EC consistent', 'EC not consistent'])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15) 
    #ax.set_yticks([0, 1])
    #ax.set_yticklabels(['Group 1', 'Group 2'])
    ax.set_ylim(0, 30)  # 添加 ylim 到所有子图中
    ax.set_title(titles[i]) 
ax=axes[0]
ax.set_ylabel('GO term count')
# 调整子图之间的间距
plt.tight_layout()

plt.show()


In [ ]:
file1_BP = '../local_snapshot/data/GO_analysis/BP1_pro.txt'
file1_MF = '../local_snapshot/data/GO_analysis/MF1_pro.txt'
file1_CC = '../local_snapshot/data/GO_analysis/CC1_pro.txt'

In [ ]:
data_BP = open(file1_BP,'r')
GO_pro_BP = {}
GO_uni_pro_BP = {}
GO_non_pro_BP = {}
for line in data_BP:
    tem = line.split('\t')
    
    GO_pro_BP[tem[0].replace('"', '')] = float(tem[1])
    if tem[0].replace('"', '') in uni_fun_fold_keys:
        GO_uni_pro_BP[tem[0].replace('"', '')] = float(tem[1])
    else:
        GO_non_pro_BP[tem[0].replace('"', '')] = float(tem[1])
    
data_MF = open(file1_MF,'r')
GO_pro_MF = {}
GO_uni_pro_MF = {}
GO_non_pro_MF = {}
for line in data_MF:
    tem = line.split('\t')
    
    GO_pro_MF[tem[0].replace('"', '')] = float(tem[1])
    if tem[0].replace('"', '') in uni_fun_fold_keys:
        GO_uni_pro_MF[tem[0].replace('"', '')] = float(tem[1])
    else:
        GO_non_pro_MF[tem[0].replace('"', '')] = float(tem[1])

data_CC = open(file1_CC,'r')
GO_pro_CC = {}
GO_uni_pro_CC = {}
GO_non_pro_CC = {}
for line in data_CC:
    tem = line.split('\t')
    
    GO_pro_CC[tem[0].replace('"', '')] = float(tem[1])
    if tem[0].replace('"', '') in uni_fun_fold_keys:
        GO_uni_pro_CC[tem[0].replace('"', '')] = float(tem[1])
    else:
        GO_non_pro_CC[tem[0].replace('"', '')] = float(tem[1])


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 两个数据列表
data1 = list(GO_pro_BP.values())
data2 = list(GO_pro_MF.values())
data3 = list(GO_pro_CC.values())
# 使用 Seaborn 绘制两个响铃图在同一个图上
ax = sns.violinplot(data=[data1, data2,data3], palette="muted", orient='v')

#plt.title('Significant GO Enrichment Proportion across Clusters')
#plt.xlabel('Density')
plt.ylabel('Proportion')
plt.grid(False)
# 设置 x 轴的刻度和标签
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['BP', 'MF','CC'])
#plt.xticks(rotation=15)
#plt.ylim(0, 30) 
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 分组数据
data_group1 = [list(GO_uni_pro_BP.values()), list(GO_non_pro_BP.values())]
data_group2 = [list(GO_uni_pro_MF.values()), list(GO_non_pro_MF.values())]
data_group3 = [list(GO_uni_pro_CC.values()), list(GO_non_pro_CC.values())]

# 创建一个包含3个子图的图
fig, axes = plt.subplots(1, 3)
titles = ['BP','MF','CC']
# 绘制每个子图的小提琴图
for i, data_group in enumerate([data_group1, data_group2, data_group3]):
    ax = sns.violinplot(data=data_group, palette="muted", orient='v', ax=axes[i])
    #ax.set_xlabel('Proportion below 0.05')
    #ax.set_ylabel('Groups')
    ax.grid(False)
    ax.set_xticklabels(['EC consistent', 'EC not consistent'])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15) 
    #ax.set_yticks([0, 1])
    #ax.set_yticklabels(['Group 1', 'Group 2'])
      # 添加 ylim 到所有子图中
    ax.set_title(titles[i]) 
ax=axes[0]
ax.set_ylabel('Proportion')
# 调整子图之间的间距
plt.tight_layout()

plt.show()


In [ ]:
file = open('../local_snapshot/data/clu_key_10','w')
for key in fun_fold.keys():
    if len(fun_fold[key])> 10:
        file.write(key+'\n')

In [ ]:
file.close()